# 4-stage pipeline comparison (train-only split)

This notebook creates a **train/test split from `data/train/` only** and compares 4 pipeline variants:

1) **Baseline (Inference Only)**: pretrained backbone embeddings + cosine clustering edges + COLMAP (SIFT) SfM
2) **Finetuned DINO Embeddings**: fine-tuned retrieval embeddings + cosine edges + COLMAP SfM
3) **Finetuned + Learned Edge Scoring**: fine-tuned embeddings + EdgeMLP edge scoring + COLMAP SfM
4) **Full Pipeline + Diffusion**: (3) + diffusion matcher for SfM

Holdout ("test") datasets:
- `ETs`
- `pt_sacrecoeur_trevi_tajmahal`
- `pt_stpeters_stpauls`

Metrics + visuals:
- **Clustering-only**: mAA / Clustering Score / Final (harmonic mean)
- **Pose-aware (IMC-style)**: pose-mAA / Clustering Score / Final (harmonic mean), using `data/train_thresholds.csv`
- Pose coverage (% images with predicted pose)

Note: stages (2)–(4) require a retrieval checkpoint; stages (3)–(4) require an EdgeMLP checkpoint.

In [1]:
from __future__ import annotations

import csv
import json
import os
import shlex
import struct
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm


def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "src").exists() and (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root (expected src/ and pyproject.toml)")


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

pio.templates.default = "plotly_white"


def run(cmd: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", shlex.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=str(REPO_ROOT), env=env)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


REPO_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025
device: cuda


## 1) Create internal split (from `data/train/` only)

In [2]:
DATA_TRAIN_ROOT = REPO_ROOT / "data" / "train"
TRAIN_LABELS = REPO_ROOT / "data" / "train_labels.csv"
TRAIN_THRESHOLDS = REPO_ROOT / "data" / "train_thresholds.csv"

TEST_DATASETS = [
    "ETs",
    "pt_sacrecoeur_trevi_tajmahal",
    "pt_stpeters_stpauls",
]
TEST_DATASETS_SET = set(TEST_DATASETS)

# Reuse the same split tag as notebook 11.
SPLIT_TAG = "holdout_ets_sacrecoeur_stpeters"
TRAIN_SPLIT_ROOT = REPO_ROOT / "data" / f"train_split_{SPLIT_TAG}"
TEST_SPLIT_ROOT = REPO_ROOT / "data" / f"test_split_{SPLIT_TAG}"
TRAIN_SPLIT_LABELS = REPO_ROOT / "data" / f"train_split_labels_{SPLIT_TAG}.csv"
TEST_SPLIT_LABELS = REPO_ROOT / "data" / f"test_split_labels_{SPLIT_TAG}.csv"

OVERWRITE_SPLIT = False


def _rm_tree(p: Path) -> None:
    if not p.exists() and not p.is_symlink():
        return
    if p.is_symlink() or p.is_file():
        p.unlink()
        return
    for q in sorted(p.rglob("*"), reverse=True):
        if q.is_symlink() or q.is_file():
            q.unlink()
        elif q.is_dir():
            q.rmdir()
    p.rmdir()


def _symlink_dir(src: Path, dst: Path, *, overwrite: bool) -> None:
    if dst.exists() or dst.is_symlink():
        if not overwrite:
            return
        _rm_tree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src, target_is_directory=True)


assert DATA_TRAIN_ROOT.exists(), f"Missing {DATA_TRAIN_ROOT}"
assert TRAIN_LABELS.exists(), f"Missing {TRAIN_LABELS}"
assert TRAIN_THRESHOLDS.exists(), f"Missing {TRAIN_THRESHOLDS}"

all_datasets = sorted([p.name for p in DATA_TRAIN_ROOT.iterdir() if p.is_dir()])
train_datasets = [d for d in all_datasets if d not in TEST_DATASETS_SET]
test_datasets = [d for d in all_datasets if d in TEST_DATASETS_SET]

TRAIN_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
TEST_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
for ds in train_datasets:
    _symlink_dir(DATA_TRAIN_ROOT / ds, TRAIN_SPLIT_ROOT / ds, overwrite=OVERWRITE_SPLIT)
for ds in test_datasets:
    _symlink_dir(DATA_TRAIN_ROOT / ds, TEST_SPLIT_ROOT / ds, overwrite=OVERWRITE_SPLIT)

labels = pd.read_csv(TRAIN_LABELS)
labels_train = labels[labels["dataset"].isin(train_datasets)].copy()
labels_test = labels[labels["dataset"].isin(test_datasets)].copy()
if OVERWRITE_SPLIT or (not TRAIN_SPLIT_LABELS.exists()):
    labels_train.to_csv(TRAIN_SPLIT_LABELS, index=False)
if OVERWRITE_SPLIT or (not TEST_SPLIT_LABELS.exists()):
    labels_test.to_csv(TEST_SPLIT_LABELS, index=False)

(REPO_ROOT / "data" / f"split_info_{SPLIT_TAG}.json").write_text(
    json.dumps(
        {
            "split_tag": SPLIT_TAG,
            "source": str(DATA_TRAIN_ROOT),
            "train_root": str(TRAIN_SPLIT_ROOT),
            "test_root": str(TEST_SPLIT_ROOT),
            "train_labels": str(TRAIN_SPLIT_LABELS),
            "test_labels": str(TEST_SPLIT_LABELS),
            "train_datasets": train_datasets,
            "test_datasets": test_datasets,
        },
        indent=2,
    )
)

print("[ok] split ready")
print("train datasets:", train_datasets)
print("test datasets:", test_datasets)


[ok] split ready
train datasets: ['amy_gardens', 'fbk_vineyard', 'imc2023_haiper', 'imc2023_heritage', 'imc2023_theather_imc2024_church', 'imc2024_dioscuri_baalshamin', 'imc2024_lizard_pond', 'pt_brandenburg_british_buckingham', 'pt_piazzasanmarco_grandplace', 'stairs']
test datasets: ['ETs', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls']


In [3]:
# Quick split summary
df = pd.read_csv(TRAIN_LABELS)
df["split"] = np.where(df["dataset"].isin(test_datasets), "holdout_test", "train")
summary = (
    df.groupby(["split", "dataset"], as_index=False)
    .agg(n_images=("image", "count"), n_scenes=("scene", "nunique"))
)
fig = px.bar(summary, x="dataset", y="n_images", color="split", barmode="group", title="Split: images per dataset")
fig.show()
summary


,split,dataset,n_images,n_scenes
0,holdout_test,ETs,22,3
1,holdout_test,pt_sacrecoeur_trevi_tajmahal,225,3
2,holdout_test,pt_stpeters_stpauls,200,2
3,train,amy_gardens,200,1
4,train,fbk_vineyard,163,3
5,train,imc2023_haiper,54,3
6,train,imc2023_heritage,209,4
7,train,imc2023_theather_imc2024_church,76,2
8,train,imc2024_dioscuri_baalshamin,138,3
9,train,imc2024_lizard_pond,214,3


## 2) Common configuration

In [4]:
MODEL_ID = "facebook/dinov2-small"

# Optional Hugging Face mirror for restricted networks.
HF_ENDPOINT = None  # e.g. "https://hf-mirror.com"

# Avoid multiprocessing DataLoader issues in notebooks.
NUM_WORKERS = 0

ENV = os.environ.copy()
if HF_ENDPOINT:
    ENV["HF_ENDPOINT"] = str(HF_ENDPOINT)

OUT_ROOT = REPO_ROOT / "outputs" / f"pipeline_compare_{SPLIT_TAG}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_ROOT


PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters')

## 3) Stage 1 cache: pretrained (inference-only) embeddings

We build a retrieval-style cache with `embeddings.npy` + `meta.json` (no fine-tuned checkpoint required).

In [5]:
from imc25.train.retrieval_transforms import load_transform_spec, make_eval_transform
from transformers import AutoModel


CACHE_PRETRAINED_TEST = OUT_ROOT / "cache_pretrained_test"
CACHE_PRETRAINED_TEST.mkdir(parents=True, exist_ok=True)


class _PathDataset(torch.utils.data.Dataset):
    def __init__(self, paths: list[Path], *, transform) -> None:
        self.paths = paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image

        p = self.paths[int(idx)]
        with Image.open(p) as im:
            im = im.convert("RGB")
            x = self.transform(im)
        return x, str(p)


def _list_images(root: Path) -> list[Path]:
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
    return [p for p in sorted(root.iterdir()) if p.is_file() and p.suffix.lower() in exts]


@torch.inference_mode()
def _embed_pretrained_backbone(paths: list[Path], *, batch_size: int) -> np.ndarray:
    if HF_ENDPOINT:
        os.environ["HF_ENDPOINT"] = str(HF_ENDPOINT)
    spec = load_transform_spec(model_id=MODEL_ID, hf_endpoint=HF_ENDPOINT)
    tf = make_eval_transform(spec)

    model = AutoModel.from_pretrained(MODEL_ID)
    model.eval().to(DEVICE)

    ds = _PathDataset(paths, transform=tf)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(NUM_WORKERS),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

    zs: list[np.ndarray] = []
    for xb, _ in tqdm(loader, desc="embed(pretrained)", leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        out = model(pixel_values=xb)
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            x = out.pooler_output
        else:
            x = out.last_hidden_state[:, 0]
        z = F.normalize(x.float(), p=2, dim=1).cpu().numpy()
        zs.append(z.astype(np.float32, copy=False))
    return np.concatenate(zs, axis=0) if zs else np.zeros((0, 1), dtype=np.float32)


def build_cache_pretrained(*, data_root: Path, cache_root: Path, datasets: list[str], overwrite: bool = False) -> None:
    cache_root.mkdir(parents=True, exist_ok=True)
    for dataset in datasets:
        ds_dir = data_root / dataset
        out_dir = cache_root / dataset
        out_dir.mkdir(parents=True, exist_ok=True)
        meta_path = out_dir / "meta.json"
        emb_path = out_dir / "embeddings.npy"
        if meta_path.exists() and emb_path.exists() and not overwrite:
            print(f"[skip] pretrained cache exists: {dataset} -> {out_dir}")
            continue
        paths = _list_images(ds_dir)
        if not paths:
            print(f"[skip] no images: {ds_dir}")
            continue
        image_ids = [p.stem for p in paths]
        emb = _embed_pretrained_backbone(paths, batch_size=64)
        np.save(emb_path, emb.astype(np.float32, copy=False))
        meta = {
            "dataset": dataset,
            "backend": "pretrained_backbone",
            "model": MODEL_ID,
            "embed_dim": int(emb.shape[1]) if emb.ndim == 2 else None,
            "image_ids": image_ids,
            "paths": [str(p.resolve()) for p in paths],
        }
        meta_path.write_text(json.dumps(meta, indent=2))
        print(f"[ok] pretrained cache: {dataset} embeddings={emb.shape} -> {out_dir}")


BUILD_STAGE1_CACHE = True
if BUILD_STAGE1_CACHE:
    build_cache_pretrained(data_root=TEST_SPLIT_ROOT, cache_root=CACHE_PRETRAINED_TEST, datasets=test_datasets, overwrite=False)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


embed(pretrained):   0%|          | 0/1 [00:00<?, ?it/s]

[ok] pretrained cache: ETs embeddings=(22, 384) -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_pretrained_test/ETs


embed(pretrained):   0%|          | 0/4 [00:00<?, ?it/s]

[ok] pretrained cache: pt_sacrecoeur_trevi_tajmahal embeddings=(225, 384) -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_pretrained_test/pt_sacrecoeur_trevi_tajmahal


embed(pretrained):   0%|          | 0/4 [00:00<?, ?it/s]

[ok] pretrained cache: pt_stpeters_stpauls embeddings=(200, 384) -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_pretrained_test/pt_stpeters_stpauls


## 4) Stage 2–4: fine-tuned retrieval embeddings + caches

We reuse an existing split checkpoint if available; otherwise you can train one.

In [6]:
# Prefer the existing split checkpoint if present (created by earlier notebook runs).
EXISTING_SPLIT_CKPT = REPO_ROOT / "outputs" / "retrieval_finetune_split" / "best.pt"
TRAIN_OUTPUT_DIR = OUT_ROOT / "retrieval_finetune"
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FINETUNED = TRAIN_OUTPUT_DIR / "best.pt"

if EXISTING_SPLIT_CKPT.exists():
    CHECKPOINT_TO_USE = EXISTING_SPLIT_CKPT
    print("[use] existing checkpoint:", CHECKPOINT_TO_USE)
else:
    CHECKPOINT_TO_USE = CHECKPOINT_FINETUNED
    print("[use] notebook checkpoint path:", CHECKPOINT_TO_USE)

RUN_TRAINING = False  # set True to force training into TRAIN_OUTPUT_DIR

if CHECKPOINT_TO_USE == CHECKPOINT_FINETUNED and (RUN_TRAINING or (not CHECKPOINT_FINETUNED.exists())):
    cmd = [
        sys.executable,
        "scripts/train_retrieval.py",
        "--data-root",
        str(TRAIN_SPLIT_ROOT),
        "--labels-csv",
        str(TRAIN_SPLIT_LABELS),
        "--output-dir",
        str(TRAIN_OUTPUT_DIR),
        "--model-id",
        str(MODEL_ID),
        "--embed-dim",
        "256",
        "--epochs",
        "6",
        "--batch-size",
        "48",
        "--num-workers",
        str(int(NUM_WORKERS)),
        "--freeze-backbone",
        "--no-include-outliers",
    ]
    if HF_ENDPOINT:
        cmd += ["--hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=ENV)
else:
    print("[skip] training")

assert CHECKPOINT_TO_USE.exists(), f"Missing checkpoint: {CHECKPOINT_TO_USE}"
CHECKPOINT_TO_USE


[use] existing checkpoint: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/retrieval_finetune_split/best.pt
[skip] training


PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/retrieval_finetune_split/best.pt')

In [7]:
CACHE_FINETUNED_TRAIN = OUT_ROOT / "cache_finetuned_train"
CACHE_FINETUNED_TEST = OUT_ROOT / "cache_finetuned_test"

BUILD_FINETUNED_CACHES = True
TOPK = 30
MUTUAL = True

if BUILD_FINETUNED_CACHES:
    # Train cache
    cmd = [
        sys.executable,
        "scripts/build_retrieval_cache.py",
        "--data-root",
        str(TRAIN_SPLIT_ROOT),
        "--cache-root",
        str(CACHE_FINETUNED_TRAIN),
        "--checkpoint",
        str(CHECKPOINT_TO_USE),
        "--topk",
        str(int(TOPK)),
        "--num-workers",
        str(int(NUM_WORKERS)),
    ]
    cmd += ["--mutual"] if MUTUAL else ["--no-mutual"]
    if HF_ENDPOINT:
        cmd += ["--hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=ENV)

    # Test cache
    cmd = [
        sys.executable,
        "scripts/build_retrieval_cache.py",
        "--data-root",
        str(TEST_SPLIT_ROOT),
        "--cache-root",
        str(CACHE_FINETUNED_TEST),
        "--checkpoint",
        str(CHECKPOINT_TO_USE),
        "--topk",
        str(int(TOPK)),
        "--num-workers",
        str(int(NUM_WORKERS)),
    ]
    cmd += ["--mutual"] if MUTUAL else ["--no-mutual"]
    if HF_ENDPOINT:
        cmd += ["--hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=ENV)


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/build_retrieval_cache.py --data-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_split_holdout_ets_sacrecoeur_stpeters --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_train --checkpoint /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/retrieval_finetune_split/best.pt --topk 30 --num-workers 0 --mutual


[ok] amy_gardens: embeddings=(200, 256) pairs=1361 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_train/amy_gardens


[ok] fbk_vineyard: embeddings=(163, 256) pairs=1524 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_train/fbk_vineyard


[ok] imc2023_haiper: embeddings=(54, 256) pairs=718 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_train/imc2023_haiper


[ok] imc2023_heritage: embeddings=(209, 256) pairs=2253 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_train/imc2023_heritage


...

embed:   0%|          | 0/4 [00:00<?, ?it/s]        

[ok] ETs: embeddings=(22, 256) pairs=231 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_test/ETs


embed:   0%|          | 0/4 [00:00<?, ?it/s]        

[ok] pt_sacrecoeur_trevi_tajmahal: embeddings=(225, 256) pairs=2122 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_test/pt_sacrecoeur_trevi_tajmahal


[ok] pt_stpeters_stpauls: embeddings=(200, 256) pairs=1758 -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_finetuned_test/pt_stpeters_stpauls


## 5) Stage 3–4: EdgeMLP (learned edge scoring) on fine-tuned embeddings

This trains a small MLP to score KNN edges as same-scene vs different-scene.

In [8]:
from imc25.models.edge_mlp import EdgeMLP


EDGE_MODELS_DIR = OUT_ROOT / "edge_models"
EDGE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
EDGE_MLP_CKPT = EDGE_MODELS_DIR / "edge_mlp.pt"

# You can also point this to an existing model, e.g. REPO_ROOT/"outputs/edge_models/edge_mlp.pt".
RUN_EDGE_MLP_TRAINING = False

EDGE_K = 30
EDGE_VAL_RATIO = 0.1
EDGE_EPOCHS = 12
EDGE_BATCH_SIZE = 2048
EDGE_LR = 1e-3
EDGE_SEED = 42


def _normalize_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    denom = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / denom


def _load_cache_embeddings(cache_root: Path, dataset: str) -> tuple[list[str], np.ndarray]:
    meta = json.loads((cache_root / dataset / "meta.json").read_text())
    image_ids = [str(x) for x in meta.get("image_ids", [])]
    emb = np.load(cache_root / dataset / "embeddings.npy")
    if len(image_ids) != int(emb.shape[0]):
        raise ValueError(f"{dataset}: meta/image_ids mismatch embeddings rows")
    return image_ids, emb.astype(np.float32, copy=False)


def _split_indices(n: int, *, val_ratio: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(int(seed))
    idx = np.arange(int(n))
    rng.shuffle(idx)
    n_val = int(round(float(val_ratio) * int(n)))
    val = np.sort(idx[:n_val])
    tr = np.sort(idx[n_val:])
    return tr, val


def _build_edge_dataset_for_dataset(*, dataset: str, cache_root: Path, labels_csv: Path, k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    image_ids, emb = _load_cache_embeddings(cache_root, dataset)
    emb = _normalize_rows(emb)

    gt = pd.read_csv(labels_csv)
    gt = gt[gt["dataset"] == dataset].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)
    scene_of = {str(r.image_key): str(r.scene) for r in gt.itertuples(index=False)}
    scene = [scene_of.get(str(i), "(missing)") for i in image_ids]

    n = int(emb.shape[0])
    if n <= 1:
        return np.zeros((0,), np.int32), np.zeros((0,), np.int32), np.zeros((0,), np.float32), emb

    kk = int(min(int(k) + 1, n))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine")
    nn.fit(emb)
    _, ind = nn.kneighbors(emb, return_distance=True)
    neigh = ind[:, 1:]

    i_list: list[int] = []
    j_list: list[int] = []
    y_list: list[float] = []
    for i in range(n):
        si = str(scene[i])
        for j in neigh[i].tolist():
            j = int(j)
            if i == j:
                continue
            sj = str(scene[j])
            # Treat any outlier involvement as negative.
            if si.lower() == "outliers" or sj.lower() == "outliers":
                y = 0.0
            else:
                y = 1.0 if si == sj else 0.0
            i_list.append(i)
            j_list.append(j)
            y_list.append(float(y))

    return (
        np.asarray(i_list, dtype=np.int32),
        np.asarray(j_list, dtype=np.int32),
        np.asarray(y_list, dtype=np.float32),
        emb,
    )


def train_edge_mlp(*, cache_root: Path, labels_csv: Path, datasets: list[str], out_path: Path) -> Path:
    all_i: list[np.ndarray] = []
    all_j: list[np.ndarray] = []
    all_y: list[np.ndarray] = []
    all_emb: list[np.ndarray] = []

    offset = 0
    for ds in datasets:
        i, j, y, emb = _build_edge_dataset_for_dataset(dataset=ds, cache_root=cache_root, labels_csv=labels_csv, k=EDGE_K)
        all_i.append(i + offset)
        all_j.append(j + offset)
        all_y.append(y)
        all_emb.append(emb)
        offset += int(emb.shape[0])

    Z = np.concatenate(all_emb, axis=0) if all_emb else np.zeros((0, 256), dtype=np.float32)
    I = np.concatenate(all_i, axis=0) if all_i else np.zeros((0,), dtype=np.int32)
    J = np.concatenate(all_j, axis=0) if all_j else np.zeros((0,), dtype=np.int32)
    Y = np.concatenate(all_y, axis=0) if all_y else np.zeros((0,), dtype=np.float32)

    if len(Z) == 0 or len(I) == 0:
        raise RuntimeError("No edges/embeddings available to train EdgeMLP")

    tr_idx, va_idx = _split_indices(len(Z), val_ratio=EDGE_VAL_RATIO, seed=EDGE_SEED)
    tr_set = set(tr_idx.tolist())
    va_set = set(va_idx.tolist())
    tr_mask = np.array([(int(i) in tr_set and int(j) in tr_set) for i, j in zip(I, J, strict=False)], dtype=bool)
    va_mask = np.array([(int(i) in va_set and int(j) in va_set) for i, j in zip(I, J, strict=False)], dtype=bool)

    I_tr, J_tr, Y_tr = I[tr_mask], J[tr_mask], Y[tr_mask]
    I_va, J_va, Y_va = I[va_mask], J[va_mask], Y[va_mask]
    print(f"EdgeMLP edges: train={len(I_tr)} val={len(I_va)} total={len(I)}")

    z_t = torch.from_numpy(Z).to(DEVICE)
    model = EdgeMLP(int(Z.shape[1])).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=float(EDGE_LR))
    loss_fn = nn.BCELoss()

    best_auc = -1.0
    best_state = None
    rng = np.random.default_rng(EDGE_SEED)

    for epoch in range(1, int(EDGE_EPOCHS) + 1):
        model.train()
        order = rng.permutation(len(I_tr))
        losses = []
        for start in range(0, len(order), int(EDGE_BATCH_SIZE)):
            idx = order[start : start + int(EDGE_BATCH_SIZE)]
            ii = torch.from_numpy(I_tr[idx]).to(DEVICE)
            jj = torch.from_numpy(J_tr[idx]).to(DEVICE)
            yb = torch.from_numpy(Y_tr[idx]).to(DEVICE)

            opt.zero_grad(set_to_none=True)
            pred = model(z_t[ii], z_t[jj])
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu().item()))

        model.eval()
        with torch.no_grad():
            if len(I_va) > 0:
                ii = torch.from_numpy(I_va).to(DEVICE)
                jj = torch.from_numpy(J_va).to(DEVICE)
                pv = model(z_t[ii], z_t[jj]).detach().cpu().numpy()
                auc = float(roc_auc_score(Y_va, pv)) if len(np.unique(Y_va)) > 1 else 0.0
                ap = float(average_precision_score(Y_va, pv)) if len(np.unique(Y_va)) > 1 else 0.0
            else:
                auc, ap = 0.0, 0.0
        print(f"Epoch {epoch:02d} | loss={np.mean(losses) if losses else 0:.4f} | val_auc={auc:.4f} | val_ap={ap:.4f}")
        if auc >= best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        best_state = model.state_dict()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": best_state, "dim": int(Z.shape[1]), "K": int(EDGE_K)}, out_path)
    print(f"[ok] saved EdgeMLP: {out_path} (best_auc={best_auc:.4f})")
    return out_path


if RUN_EDGE_MLP_TRAINING or (not EDGE_MLP_CKPT.exists()):
    train_edge_mlp(cache_root=CACHE_FINETUNED_TRAIN, labels_csv=TRAIN_SPLIT_LABELS, datasets=train_datasets, out_path=EDGE_MLP_CKPT)
else:
    print("[skip] EdgeMLP exists:", EDGE_MLP_CKPT)

assert EDGE_MLP_CKPT.exists(), f"Missing EdgeMLP checkpoint: {EDGE_MLP_CKPT}"
EDGE_MLP_CKPT


EdgeMLP edges: train=36455 val=473 total=44940
Epoch 01 | loss=0.6610 | val_auc=0.3241 | val_ap=0.8032
Epoch 02 | loss=0.5252 | val_auc=0.3313 | val_ap=0.8048
Epoch 03 | loss=0.4504 | val_auc=0.8028 | val_ap=0.9698
Epoch 04 | loss=0.3726 | val_auc=0.8350 | val_ap=0.9768
Epoch 05 | loss=0.3174 | val_auc=0.8752 | val_ap=0.9836
Epoch 06 | loss=0.2823 | val_auc=0.9132 | val_ap=0.9892
Epoch 07 | loss=0.2507 | val_auc=0.9436 | val_ap=0.9932
Epoch 08 | loss=0.2184 | val_auc=0.9639 | val_ap=0.9957
Epoch 09 | loss=0.1811 | val_auc=0.9809 | val_ap=0.9977
Epoch 10 | loss=0.1547 | val_auc=0.9832 | val_ap=0.9980
Epoch 11 | loss=0.1404 | val_auc=0.9854 | val_ap=0.9983
Epoch 12 | loss=0.1320 | val_auc=0.9858 | val_ap=0.9983
[ok] saved EdgeMLP: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp.pt (best_auc=0.9858)


PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp.pt')

## 6) Run the 4 stages (cluster → SfM → export predictions)

Outputs:
- clusters CSV per stage
- reconstruction folders per stage
- prediction CSV per stage (cluster label + pose per image)


In [9]:
@dataclass(frozen=True)
class ClusterCfg:
    topk: int = 30
    mutual: bool = True
    min_sim: float = 0.35
    method: str = "louvain"  # louvain|greedy|components
    min_cluster_size: int = 3
    edge_scorer: str = "cosine"  # cosine|mlp|gbdt
    edge_model: Path | None = None
    edge_topm: int | None = None


def run_cluster_retrieval(*, cache_root: Path, out_csv: Path, datasets: list[str], cfg: ClusterCfg) -> None:
    cmd = [
        sys.executable,
        "scripts/cluster_retrieval.py",
        "--cache-root",
        str(cache_root),
        "--out-csv",
        str(out_csv),
        "--topk",
        str(int(cfg.topk)),
        "--min-sim",
        str(float(cfg.min_sim)),
        "--method",
        str(cfg.method),
        "--min-cluster-size",
        str(int(cfg.min_cluster_size)),
        "--edge-scorer",
        str(cfg.edge_scorer),
    ]
    for ds in datasets:
        cmd += ["--dataset", str(ds)]
    if cfg.edge_scorer in {"mlp", "gbdt"}:
        if cfg.edge_model is None:
            raise ValueError("edge_model required when edge_scorer is mlp/gbdt")
        cmd += ["--edge-model", str(cfg.edge_model)]
        if cfg.edge_topm is not None:
            cmd += ["--edge-topm", str(int(cfg.edge_topm))]
    cmd += ["--mutual"] if cfg.mutual else ["--no-mutual"]
    run(cmd, env=ENV)


def run_recon(*, clusters_csv: Path, cache_root: Path, data_root: Path, out_root: Path, matcher: str, runner: str = "auto", overwrite: bool = False) -> None:
    cmd = [
        sys.executable,
        "scripts/colmap_dense_clusters.py",
        "--clusters-csv",
        str(clusters_csv),
        "--cache-root",
        str(cache_root),
        "--data-root",
        str(data_root),
        "--output-root",
        str(out_root),
        "--runner",
        str(runner),
        "--matcher",
        str(matcher),
        "--sparse-only",
    ]
    if overwrite:
        cmd += ["--overwrite"]
    if matcher == "diffusion" and HF_ENDPOINT:
        cmd += ["--diffusion-hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=ENV)


STAGE1_CLUSTERS = RESULTS_DIR / "stage1_clusters.csv"
STAGE2_CLUSTERS = RESULTS_DIR / "stage2_clusters.csv"
STAGE3_CLUSTERS = RESULTS_DIR / "stage3_clusters.csv"
STAGE4_CLUSTERS = RESULTS_DIR / "stage4_clusters.csv"

CFG1 = ClusterCfg(edge_scorer="cosine", min_sim=0.35, topk=30, mutual=True, method="louvain")
CFG2 = ClusterCfg(edge_scorer="cosine", min_sim=0.35, topk=30, mutual=True, method="louvain")
CFG3 = ClusterCfg(edge_scorer="mlp", edge_model=EDGE_MLP_CKPT, edge_topm=5, min_sim=0.55, topk=30, mutual=True, method="louvain")

RUN_CLUSTERING = True
if RUN_CLUSTERING:
    run_cluster_retrieval(cache_root=CACHE_PRETRAINED_TEST, out_csv=STAGE1_CLUSTERS, datasets=test_datasets, cfg=CFG1)
    run_cluster_retrieval(cache_root=CACHE_FINETUNED_TEST, out_csv=STAGE2_CLUSTERS, datasets=test_datasets, cfg=CFG2)
    run_cluster_retrieval(cache_root=CACHE_FINETUNED_TEST, out_csv=STAGE3_CLUSTERS, datasets=test_datasets, cfg=CFG3)
    # Stage 4 uses the same clustering as stage 3 (only reconstruction changes)
    Path(STAGE4_CLUSTERS).write_text(Path(STAGE3_CLUSTERS).read_text())

STAGE1_RECON = OUT_ROOT / "recon_stage1_colmap"
STAGE2_RECON = OUT_ROOT / "recon_stage2_colmap"
STAGE3_RECON = OUT_ROOT / "recon_stage3_colmap"
STAGE4_RECON = OUT_ROOT / "recon_stage4_diffusion"

RUN_SFM = False  # set True to run reconstructions (can take time)
OVERWRITE_SFM = False
RUNNER = "auto"  # auto|docker|local

if RUN_SFM:
    run_recon(clusters_csv=STAGE1_CLUSTERS, cache_root=CACHE_PRETRAINED_TEST, data_root=TEST_SPLIT_ROOT, out_root=STAGE1_RECON, matcher="colmap", runner=RUNNER, overwrite=OVERWRITE_SFM)
    run_recon(clusters_csv=STAGE2_CLUSTERS, cache_root=CACHE_FINETUNED_TEST, data_root=TEST_SPLIT_ROOT, out_root=STAGE2_RECON, matcher="colmap", runner=RUNNER, overwrite=OVERWRITE_SFM)
    run_recon(clusters_csv=STAGE3_CLUSTERS, cache_root=CACHE_FINETUNED_TEST, data_root=TEST_SPLIT_ROOT, out_root=STAGE3_RECON, matcher="colmap", runner=RUNNER, overwrite=OVERWRITE_SFM)
    run_recon(clusters_csv=STAGE4_CLUSTERS, cache_root=CACHE_FINETUNED_TEST, data_root=TEST_SPLIT_ROOT, out_root=STAGE4_RECON, matcher="diffusion", runner=RUNNER, overwrite=OVERWRITE_SFM)
else:
    print("[skip] SfM (RUN_SFM=False)")


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/cluster_retrieval.py --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/cache_pretrained_test --out-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_sacrecoeur_stpeters/results/stage1_clusters.csv --topk 30 --min-sim 0.35 --method louvain --min-cluster-size 3 --edge-scorer cosine --dataset ETs --dataset pt_sacrecoeur_trevi_tajmahal --dataset pt_stpeters_stpauls --mutual
Inference embeddings shape: (22, 384)
[ok] ETs: clusters=2 outliers=0 images=22
Inference embeddings shape: (225, 384)
[ok] pt_sacrecoeur_trevi_tajmahal: clusters=6 outliers=4 images=225
Inference embeddings shape: (200, 384)
[ok] pt_stpeters_stpauls: clusters=7 outliers=3 images=200
[ok] wrote /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/pipeline_compare_holdout_ets_s

### Export predictions (cluster + pose) from recon outputs

This creates one CSV per stage with columns:
`dataset, scene, image, rotation_matrix, translation_vector`.


In [10]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def _list_images_in_dataset(ds_dir: Path) -> list[Path]:
    return [p for p in sorted(ds_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMAGE_EXTS]


def _u64_head(path: Path) -> int:
    with path.open("rb") as f:
        head = f.read(8)
    if len(head) != 8:
        return 0
    (n,) = struct.unpack("<Q", head)
    return int(n)


def _pick_best_sparse_model(sparse_root: Path) -> Path | None:
    if not sparse_root.exists():
        return None
    candidates = [p for p in sparse_root.iterdir() if p.is_dir() and p.name.isdigit()]
    if not candidates:
        return None
    best_score: tuple[int, int, int] | None = None
    best_path: Path | None = None
    for p in sorted(candidates, key=lambda x: int(x.name)):
        n_images = _u64_head(p / "images.bin") if (p / "images.bin").exists() else 0
        n_points = _u64_head(p / "points3D.bin") if (p / "points3D.bin").exists() else 0
        score = (int(n_images), int(n_points), int(p.name))
        if best_score is None or score > best_score:
            best_score = score
            best_path = p
    return best_path


def _read_c_string(f) -> str:
    buf = bytearray()
    while True:
        b = f.read(1)
        if b == b"":
            raise EOFError("Unexpected EOF while reading string")
        if b == b"\x00":
            break
        buf.extend(b)
    return buf.decode("utf-8", errors="replace")


def _read_colmap_images_bin(path: Path) -> dict[str, tuple[tuple[float, float, float, float], tuple[float, float, float]]]:
    out: dict[str, tuple[tuple[float, float, float, float], tuple[float, float, float]]] = {}
    with path.open("rb") as f:
        num_images = struct.unpack("<Q", f.read(8))[0]
        for _ in range(int(num_images)):
            _image_id = struct.unpack("<i", f.read(4))[0]
            qvec = struct.unpack("<dddd", f.read(32))
            tvec = struct.unpack("<ddd", f.read(24))
            _camera_id = struct.unpack("<i", f.read(4))[0]
            name = _read_c_string(f)
            num_points2d = struct.unpack("<Q", f.read(8))[0]
            f.seek(int(num_points2d) * 24, 1)
            out[str(name)] = ((float(qvec[0]), float(qvec[1]), float(qvec[2]), float(qvec[3])), (float(tvec[0]), float(tvec[1]), float(tvec[2])))
    return out


def _qvec_to_rotmat(q: tuple[float, float, float, float]) -> list[list[float]]:
    qw, qx, qy, qz = q
    return [
        [1.0 - 2.0 * qy * qy - 2.0 * qz * qz, 2.0 * qx * qy - 2.0 * qw * qz, 2.0 * qx * qz + 2.0 * qw * qy],
        [2.0 * qx * qy + 2.0 * qw * qz, 1.0 - 2.0 * qx * qx - 2.0 * qz * qz, 2.0 * qy * qz - 2.0 * qw * qx],
        [2.0 * qx * qz - 2.0 * qw * qy, 2.0 * qy * qz + 2.0 * qw * qx, 1.0 - 2.0 * qx * qx - 2.0 * qy * qy],
    ]


def _fmt_floats(vals: Iterable[float]) -> str:
    return ";".join(["nan" if (v != v) else f"{float(v):.12g}" for v in vals])


def _nan_pose() -> tuple[str, str]:
    return (";".join(["nan"] * 9), ";".join(["nan"] * 3))


def export_predictions_with_poses(*, clusters_csv: Path, data_root: Path, recon_root: Path, out_csv: Path, outliers_label: str = "outliers") -> Path:
    clusters = pd.read_csv(clusters_csv)
    clusters = clusters[["dataset", "scene", "image_id"]].copy()
    clusters["image_key"] = clusters["image_id"].map(lambda x: Path(str(x)).stem)
    scene_of = {(str(r.dataset), str(r.image_key)): str(r.scene) for r in clusters.itertuples(index=False)}

    # Build pose index: (dataset, scene, stem) -> (R_str, t_str)
    pose_idx: dict[tuple[str, str, str], tuple[str, str]] = {}
    scenes = {(str(r.dataset), str(r.scene)) for r in clusters.itertuples(index=False) if str(r.scene).lower() != outliers_label.lower()}
    for dataset, scene in sorted(scenes):
        sparse_root = recon_root / f"{dataset}_{scene}" / "sparse"
        best = _pick_best_sparse_model(sparse_root)
        if best is None:
            continue
        images_bin = best / "images.bin"
        if not images_bin.exists():
            continue
        try:
            imgs = _read_colmap_images_bin(images_bin)
        except Exception:
            continue
        for name, (qvec, tvec) in imgs.items():
            base = PurePosixPath(str(name)).name
            stem = Path(base).stem
            R = _qvec_to_rotmat(qvec)
            R_flat = [R[0][0], R[0][1], R[0][2], R[1][0], R[1][1], R[1][2], R[2][0], R[2][1], R[2][2]]
            pose_idx[(dataset, scene, stem)] = (_fmt_floats(R_flat), _fmt_floats(tvec))

    nan_R, nan_t = _nan_pose()
    rows = []
    for ds_dir in sorted([p for p in data_root.iterdir() if p.is_dir()]):
        dataset = ds_dir.name
        for p in _list_images_in_dataset(ds_dir):
            stem = p.stem
            scene = scene_of.get((dataset, stem), outliers_label)
            if str(scene).lower() == outliers_label.lower():
                rot, trans = nan_R, nan_t
            else:
                rot, trans = pose_idx.get((dataset, str(scene), stem), (nan_R, nan_t))
            rows.append(
                {
                    "dataset": dataset,
                    "scene": str(scene),
                    "image": p.name,
                    "rotation_matrix": rot,
                    "translation_vector": trans,
                }
            )

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[ok] wrote predictions: {out_csv} rows={len(rows)}")
    return out_csv


PRED1 = RESULTS_DIR / "stage1_predictions.csv"
PRED2 = RESULTS_DIR / "stage2_predictions.csv"
PRED3 = RESULTS_DIR / "stage3_predictions.csv"
PRED4 = RESULTS_DIR / "stage4_predictions.csv"

EXPORT_PREDS = False  # requires SfM outputs
if EXPORT_PREDS:
    export_predictions_with_poses(clusters_csv=STAGE1_CLUSTERS, data_root=TEST_SPLIT_ROOT, recon_root=STAGE1_RECON, out_csv=PRED1)
    export_predictions_with_poses(clusters_csv=STAGE2_CLUSTERS, data_root=TEST_SPLIT_ROOT, recon_root=STAGE2_RECON, out_csv=PRED2)
    export_predictions_with_poses(clusters_csv=STAGE3_CLUSTERS, data_root=TEST_SPLIT_ROOT, recon_root=STAGE3_RECON, out_csv=PRED3)
    export_predictions_with_poses(clusters_csv=STAGE4_CLUSTERS, data_root=TEST_SPLIT_ROOT, recon_root=STAGE4_RECON, out_csv=PRED4)
else:
    print("[skip] export predictions (EXPORT_PREDS=False)")


[skip] export predictions (EXPORT_PREDS=False)


## 7) Metrics

We report:
- clustering-only metrics from `clusters.csv`
- pose-aware metrics from `predictions.csv` (requires SfM)


In [11]:
# --- Clustering-only metrics (mAA/CS/Final) ---

def _image_keys_in_dir(ds_dir: Path) -> set[str]:
    if not ds_dir.exists():
        return set()
    out = set()
    for p in ds_dir.iterdir():
        if (p.is_file() or p.is_symlink()) and p.suffix.lower() in IMAGE_EXTS:
            out.add(p.stem)
    return out


def load_ground_truth_clustering(*, labels_csv: Path, data_root: Path, drop_outliers_scene: bool) -> Tuple[Dict[str, Dict[str, set[str]]], Dict[str, set[str]]]:
    gt = pd.read_csv(labels_csv)
    gt = gt[["dataset", "scene", "image"]].copy()
    if drop_outliers_scene:
        gt = gt[gt["scene"].astype(str).str.lower() != "outliers"].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, scene), g in gt.groupby(["dataset", "scene"], sort=True):
        scenes.setdefault(str(dataset), {})[str(scene)] = set(g["image_key"].tolist())

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        if not keys:
            keys = set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())
        all_images[dataset] = keys

    return scenes, all_images


def load_predictions_clusters(*, pred_csv: Path, expected_images: Dict[str, set[str]], outliers_label: str = "outliers") -> Dict[str, Dict[str, set[str]]]:
    df = pd.read_csv(pred_csv)
    df = df.rename(columns={"scene": "pred_cluster"})
    df["image_key"] = df["image_id"].map(lambda x: Path(str(x)).stem)

    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        kept.append(g[g["image_key"].isin(exp)] if exp else g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # Fill missing as outliers
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        rows_missing.extend({"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing)
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    out: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, c), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        out.setdefault(str(dataset), {})[str(c)] = set(g["image_key"].tolist())
    return out


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


@dataclass(frozen=True)
class ClusteringScore:
    dataset: str
    mAA: float
    clustering_score: float
    final_score: float
    n_images_in_scenes: int
    n_correct: int
    n_cluster_images_summed: int


def score_dataset_clustering(*, dataset: str, gt_scenes: Dict[str, set[str]], pred_clusters: Dict[str, set[str]], outliers_label: str = "outliers") -> ClusteringScore:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}
    edges: list[tuple[float, float, int, int, str, str]] = []
    for scene, scene_imgs in gt_scenes.items():
        n_scene = len(scene_imgs)
        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter <= 0:
                continue
            maa = inter / n_scene
            cs = inter / len(c_imgs) if len(c_imgs) else 0.0
            edges.append((maa, cs, inter, -len(c_imgs), scene, c_label))
    edges.sort(key=lambda x: (x[0], x[1], x[2], x[3]), reverse=True)

    assigned_scene: dict[str, tuple[str, int, int]] = {}
    used_clusters: set[str] = set()
    for maa, cs, inter, neg_sz, scene, c_label in edges:
        if scene in assigned_scene or c_label in used_clusters:
            continue
        assigned_scene[scene] = (c_label, int(inter), int(-neg_sz))
        used_clusters.add(c_label)

    total_correct = 0
    total_scene_images = 0
    total_cluster_images = 0
    for scene, scene_imgs in gt_scenes.items():
        total_scene_images += len(scene_imgs)
        if scene in assigned_scene:
            _, inter, csz = assigned_scene[scene]
            total_correct += inter
            total_cluster_images += csz

    maa = (total_correct / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    return ClusteringScore(
        dataset=dataset,
        mAA=float(maa),
        clustering_score=float(cs),
        final_score=float(harmonic_mean(maa, cs)),
        n_images_in_scenes=int(total_scene_images),
        n_correct=int(total_correct),
        n_cluster_images_summed=int(total_cluster_images),
    )


def score_clusters_csv(*, clusters_csv: Path) -> pd.DataFrame:
    gt_scenes, all_images = load_ground_truth_clustering(labels_csv=TEST_SPLIT_LABELS, data_root=TEST_SPLIT_ROOT, drop_outliers_scene=True)
    pred = load_predictions_clusters(pred_csv=clusters_csv, expected_images=all_images, outliers_label="outliers")
    rows = []
    for ds in sorted(set(gt_scenes.keys()) & set(pred.keys())):
        rows.append(score_dataset_clustering(dataset=ds, gt_scenes=gt_scenes[ds], pred_clusters=pred[ds]).__dict__)
    return pd.DataFrame(rows).sort_values("dataset")


In [12]:
cluster_scores = []
for name, csv_path in [
    ("1) baseline", STAGE1_CLUSTERS),
    ("2) finetuned", STAGE2_CLUSTERS),
    ("3) finetuned+edge", STAGE3_CLUSTERS),
    ("4) +diffusion (same clusters)", STAGE4_CLUSTERS),
]:
    df_s = score_clusters_csv(clusters_csv=csv_path)
    df_s.insert(0, "stage", name)
    cluster_scores.append(df_s)

cluster_scores_df = pd.concat(cluster_scores, ignore_index=True)
cluster_scores_df.to_csv(RESULTS_DIR / "clustering_only_scores.csv", index=False)

fig = px.bar(cluster_scores_df, x="dataset", y="final_score", color="stage", barmode="group", title="Clustering-only Final Score")
fig.show()
cluster_scores_df


,stage,dataset,mAA,clustering_score,final_score,n_images_in_scenes,n_correct,n_cluster_images_summed
0,1) baseline,ETs,1.000000,0.863636,0.926829,19,19,22
1,1) baseline,pt_sacrecoeur_trevi_tajmahal,0.675556,1.000000,0.806366,225,152,152
2,1) baseline,pt_stpeters_stpauls,0.535000,1.000000,0.697068,200,107,107
3,2) finetuned,ETs,1.000000,0.863636,0.926829,19,19,22
4,2) finetuned,pt_sacrecoeur_trevi_tajmahal,0.715556,1.000000,0.834197,225,161,161
5,2) finetuned,pt_stpeters_stpauls,0.580000,1.000000,0.734177,200,116,116
6,3) finetuned+edge,ETs,0.684211,0.866667,0.764706,19,13,15
7,3) finetuned+edge,pt_sacrecoeur_trevi_tajmahal,0.715556,1.000000,0.834197,225,161,161
8,3) finetuned+edge,pt_stpeters_stpauls,0.415000,1.000000,0.586572,200,83,83
9,4) +diffusion (same clusters),ETs,0.684211,0.866667,0.764706,19,13,15


In [13]:
# --- Pose-aware metrics (pose-mAA/CS/Final) ---

def parse_floats_semicolon(s: str, expected: int) -> Optional[np.ndarray]:
    if not isinstance(s, str):
        return None
    s = s.strip()
    if not s or s.lower() == "nan":
        return None
    parts = s.split(";")
    if len(parts) != expected:
        return None
    arr = np.array([float(x) for x in parts], dtype=np.float64)
    if np.any(~np.isfinite(arr)):
        return None
    return arr


def parse_rotation(s: str) -> Optional[np.ndarray]:
    v = parse_floats_semicolon(s, 9)
    return None if v is None else v.reshape(3, 3)


def parse_translation(s: str) -> Optional[np.ndarray]:
    return parse_floats_semicolon(s, 3)


def camera_center(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    return -R.T @ t


def umeyama_similarity(X: np.ndarray, Y: np.ndarray, *, with_scale: bool = True) -> Tuple[float, np.ndarray, np.ndarray]:
    if X.shape != Y.shape or X.ndim != 2 or X.shape[1] != 3:
        raise ValueError("X and Y must have shape (N,3) and match")
    n = X.shape[0]
    if n < 3:
        raise ValueError("Need at least 3 points")
    muX = X.mean(axis=0)
    muY = Y.mean(axis=0)
    Xc = X - muX
    Yc = Y - muY
    cov = (Yc.T @ Xc) / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1
    R = U @ S @ Vt
    if with_scale:
        varX = (Xc**2).sum(axis=1).mean()
        scale = float(np.trace(np.diag(D) @ S) / varX) if varX > 0 else 1.0
    else:
        scale = 1.0
    t = muY - scale * (R @ muX)
    return scale, R, t


def estimate_similarity_ransac(X: np.ndarray, Y: np.ndarray, *, max_iters: int = 512, inlier_threshold: float = 0.1, seed: int = 0) -> Optional[Tuple[float, np.ndarray, np.ndarray, np.ndarray]]:
    if X.shape[0] < 3:
        return None
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    best = None
    best_inliers = -1
    best_median = np.inf
    idx = np.arange(n)
    for _ in range(int(max_iters)):
        sample = rng.choice(idx, size=3, replace=False)
        try:
            s, R, t = umeyama_similarity(X[sample], Y[sample], with_scale=True)
        except Exception:
            continue
        X_aligned = (s * (R @ X.T)).T + t
        err = np.linalg.norm(Y - X_aligned, axis=1)
        inliers = err < float(inlier_threshold)
        cnt = int(inliers.sum())
        med = float(np.median(err[inliers])) if cnt > 0 else np.inf
        if (cnt > best_inliers) or (cnt == best_inliers and med < best_median):
            best_inliers = cnt
            best_median = med
            best = (s, R, t, inliers)
    if best is None or best_inliers < 3:
        return None
    s0, R0, t0, inliers = best
    try:
        s, R, t = umeyama_similarity(X[inliers], Y[inliers], with_scale=True)
    except Exception:
        s, R, t = s0, R0, t0
    return s, R, t, inliers


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem
    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id`")
    s = image_id.strip()
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]
    return Path(s).stem


def load_ground_truth_pose(*, labels_csv: Path, thresholds_csv: Path, data_root: Path) -> Tuple[Dict[str, Dict[str, set[str]]], Dict[Tuple[str, str], np.ndarray], Dict[Tuple[str, str], np.ndarray], Dict[str, set[str]]]:
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"GT missing columns: {sorted(missing)}")
    gt = gt[list(required)].copy()
    gt = gt[gt["scene"].astype(str).str.lower() != "outliers"].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    centers: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in gt.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        k = str(r["image_key"])
        scenes.setdefault(ds, {}).setdefault(sc, set()).add(k)
        R = parse_rotation(str(r["rotation_matrix"]))
        t = parse_translation(str(r["translation_vector"]))
        if R is None or t is None:
            continue
        centers[(ds, k)] = camera_center(R, t)

    thr = pd.read_csv(thresholds_csv)
    thresholds: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in thr.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        parts = [float(x) for x in str(r["thresholds"]).split(";")]
        parts = [x for x in parts if x > 0]
        thresholds[(ds, sc)] = np.array(parts, dtype=np.float64)

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        all_images[dataset] = keys if keys else set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())

    return scenes, centers, thresholds, all_images


def load_predictions_with_poses(*, pred_csv: Path, expected_images: Dict[str, set[str]], outliers_label: str = "outliers") -> Tuple[Dict[str, Dict[str, set[str]]], Dict[Tuple[str, str], Optional[np.ndarray]]]:
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("pred CSV must contain dataset")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("pred CSV must contain scene/cluster/cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("pred CSV must contain image or image_id")

    has_pose = ("rotation_matrix" in cols) and ("translation_vector" in cols)

    keep_cols = ["dataset", cluster_col]
    if image_col:
        keep_cols.append(image_col)
    if image_id_col:
        keep_cols.append(image_id_col)
    if has_pose:
        keep_cols += ["rotation_matrix", "translation_vector"]

    df = df[keep_cols].copy().rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r.get(image_id_col)) else None,
            image=str(r[image_col]) if image_col and pd.notna(r.get(image_col)) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        kept.append(g[g["image_key"].isin(exp)] if exp else g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # Fill missing images as outliers.
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        rows_missing.extend({"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing)
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    clusters: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        clusters.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())

    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]] = {}
    if has_pose:
        for _, r in df.iterrows():
            ds = str(r["dataset"])
            k = str(r["image_key"])
            R = parse_rotation(str(r.get("rotation_matrix", "")))
            t = parse_translation(str(r.get("translation_vector", "")))
            pred_centers[(ds, k)] = None if (R is None or t is None) else camera_center(R, t)
    else:
        for _, r in df.iterrows():
            pred_centers[(str(r["dataset"]), str(r["image_key"]))] = None

    return clusters, pred_centers


def pose_mAA_for_scene_and_cluster(
    *,
    dataset: str,
    scene_images: set[str],
    cluster_images: set[str],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds: np.ndarray,
    min_align_points: int = 3,
    use_ransac: bool = True,
) -> Tuple[float, int]:
    inter = sorted(scene_images & cluster_images)
    if not inter:
        return 0.0, 0

    X = []
    Y = []
    for k in inter:
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        X.append(pc)
        Y.append(gc)
    if len(X) < int(min_align_points):
        return 0.0, len(X)

    X = np.stack(X, axis=0)
    Y = np.stack(Y, axis=0)

    max_thr = float(np.max(thresholds)) if thresholds.size else 0.0
    if max_thr <= 0:
        return 0.0, len(X)

    if use_ransac:
        est = estimate_similarity_ransac(X, Y, max_iters=512, inlier_threshold=max_thr, seed=0)
        if est is None:
            return 0.0, len(X)
        s, R, t, _ = est
    else:
        s, R, t = umeyama_similarity(X, Y, with_scale=True)

    # Errors for all scene images (missing poses => inf).
    errors = np.full((len(scene_images),), np.inf, dtype=np.float64)
    scene_list = sorted(scene_images)
    for i, k in enumerate(scene_list):
        if k not in cluster_images:
            continue
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        pa = (s * (R @ pc)) + t
        errors[i] = float(np.linalg.norm(gc - pa))

    accs = [(errors < thr).mean() for thr in thresholds]
    return float(np.mean(accs)), len(X)


@dataclass(frozen=True)
class PoseScore:
    dataset: str
    pose_mAA: float
    clustering_score: float
    final_score: float
    pose_coverage: float
    n_pose: int
    n_images: int


def score_dataset_pose_aware(
    *,
    dataset: str,
    gt_scenes: Dict[str, set[str]],
    pred_clusters: Dict[str, set[str]],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds_by_scene: Dict[Tuple[str, str], np.ndarray],
    expected_images: set[str],
    outliers_label: str = "outliers",
) -> PoseScore:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    total_scene_images = sum(len(v) for v in gt_scenes.values())
    total_correct = 0
    total_cluster_images = 0
    pose_mAA_weighted_sum = 0.0

    for scene, scene_imgs in gt_scenes.items():
        thresholds = thresholds_by_scene.get((dataset, scene), np.array([], dtype=np.float64))
        best_pose = -1.0
        best_cs = -1.0
        best_correct = 0
        best_cluster_size = 0
        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter <= 0:
                continue
            cs = inter / len(c_imgs) if len(c_imgs) else 0.0
            pose, _n_align = pose_mAA_for_scene_and_cluster(
                dataset=dataset,
                scene_images=scene_imgs,
                cluster_images=c_imgs,
                gt_centers=gt_centers,
                pred_centers=pred_centers,
                thresholds=thresholds,
            )
            if (pose > best_pose) or (pose == best_pose and cs > best_cs):
                best_pose = pose
                best_cs = cs
                best_correct = inter
                best_cluster_size = len(c_imgs)
        if best_pose < 0:
            best_pose = 0.0
            best_correct = 0
            best_cluster_size = 0
        pose_mAA_weighted_sum += best_pose * len(scene_imgs)
        total_correct += best_correct
        total_cluster_images += best_cluster_size

    pose_mAA = (pose_mAA_weighted_sum / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0

    n_pose = sum(pred_centers.get((dataset, k)) is not None for k in expected_images)
    n_total = len(expected_images)
    cov = (n_pose / n_total) if n_total else 0.0

    return PoseScore(
        dataset=dataset,
        pose_mAA=float(pose_mAA),
        clustering_score=float(cs),
        final_score=float(harmonic_mean(pose_mAA, cs)),
        pose_coverage=float(cov),
        n_pose=int(n_pose),
        n_images=int(n_total),
    )


def score_predictions_pose_aware(*, pred_csv: Path) -> pd.DataFrame:
    gt_scenes, gt_centers, thr_by_scene, all_images = load_ground_truth_pose(
        labels_csv=TEST_SPLIT_LABELS,
        thresholds_csv=TRAIN_THRESHOLDS,
        data_root=TEST_SPLIT_ROOT,
    )
    pred_clusters, pred_centers = load_predictions_with_poses(pred_csv=pred_csv, expected_images=all_images)
    rows = []
    for ds in sorted(set(gt_scenes.keys()) & set(pred_clusters.keys())):
        rows.append(
            score_dataset_pose_aware(
                dataset=ds,
                gt_scenes=gt_scenes[ds],
                pred_clusters=pred_clusters[ds],
                gt_centers=gt_centers,
                pred_centers=pred_centers,
                thresholds_by_scene=thr_by_scene,
                expected_images=all_images[ds],
            ).__dict__
        )
    return pd.DataFrame(rows).sort_values("dataset")


In [14]:
POSE_EVAL = False  # requires PRED*.csv to exist (set EXPORT_PREDS=True earlier)

if POSE_EVAL:
    pose_scores = []
    for stage, pred_csv in [
        ("1) baseline", PRED1),
        ("2) finetuned", PRED2),
        ("3) finetuned+edge", PRED3),
        ("4) +diffusion", PRED4),
    ]:
        df_p = score_predictions_pose_aware(pred_csv=pred_csv)
        df_p.insert(0, "stage", stage)
        pose_scores.append(df_p)

    pose_scores_df = pd.concat(pose_scores, ignore_index=True)
    pose_scores_df.to_csv(RESULTS_DIR / "pose_aware_scores.csv", index=False)

    fig = px.bar(pose_scores_df, x="dataset", y="final_score", color="stage", barmode="group", title="Pose-aware Final Score")
    fig.show()

    fig = px.bar(pose_scores_df, x="dataset", y="pose_coverage", color="stage", barmode="group", title="Pose coverage")
    fig.show()

    pose_scores_df
else:
    print("(skip) pose-aware evaluation")


(skip) pose-aware evaluation
